# 01 — The Maxwell–Boltzmann distribution, live

Give every atom in a gas *exactly the same speed*, pointing in random
directions — about as far from thermal equilibrium as a velocity
distribution gets. Then let them collide. Within a few hundred collisions
the most famous distribution in statistical mechanics assembles itself in
front of you: velocity components become Gaussian, speeds become
Maxwell–Boltzmann. No thermostat, no randomness added — just Newton's laws
(this is Boltzmann's H-theorem happening in real time).

We use a 2D gas with purely repulsive interactions (the WCA potential — a
Lennard-Jones potential cut at its minimum), so collisions redistribute
energy without any of it hiding as potential energy.

In [ ]:
%pip install lammps-js matplotlib

## A gas where every atom has the same speed

`velocity create` draws random directions; two atom-style variables then
rescale every velocity to the same magnitude $v_0 = 1.5$. In LJ units
($m = k_B = 1$) the temperature in 2D is $T = \langle v^2 angle / 2$,
so it starts at $v_0^2/2 = 1.125$ and stays within a few percent of it
(during a collision a little energy briefly sits in the repulsive
potential) while the distribution's shape changes completely.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lammps import lammps, LMP_VAR_ATOM

lmp = await lammps(output=None)
lmp.commands_string("""
units         lj
dimension     2
lattice       sq 0.3
region        box block 0 32 0 32 -0.1 0.1
create_box    1 box
create_atoms  1 box
mass          1 1.0
pair_style    lj/cut 1.122462
pair_coeff    1 1 1.0 1.0
pair_modify   shift yes

velocity      all create 1.0 777 dist gaussian
variable      v0 equal 1.5
variable      s  atom v_v0/sqrt(vx*vx+vy*vy)
variable      ux atom vx*v_s
variable      uy atom vy*v_s
velocity      all set v_ux v_uy NULL

variable      vxa atom vx
variable      spd atom sqrt(vx*vx+vy*vy)

fix           1 all nve
fix           2d all enforce2d
run           0
""")
print(lmp.get_natoms(), "atoms, all at speed 1.5, T =", round(lmp.get_thermo("temp"), 4))

## Watch equilibrium assemble

Sample the velocity components at a few moments. At $t=0$ every speed is
$v_0$, so the distribution of $v_x = v_0\cos\theta$ has spikes at
$\pm v_0$ (an arcsine distribution — most of a circle is near its
edges). Then collisions take over:

In [ ]:
snapshots = {}
for label, steps in [("t = 0", 0), ("after 100 steps", 100),
                     ("after 400 steps", 300), ("after 4000 steps", 3600)]:
    if steps:
        lmp.command(f"run {steps}")
    snapshots[label] = (lmp.extract_variable("vxa", vartype=LMP_VAR_ATOM).copy(),
                        lmp.extract_variable("spd", vartype=LMP_VAR_ATOM).copy())
T = lmp.get_thermo("temp")
print("temperature after equilibration:", round(T, 4), "(started at v0²/2 = 1.125)")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True)
v = np.linspace(-3.5, 3.5, 300)
gauss = np.exp(-v**2 / (2 * T)) / np.sqrt(2 * np.pi * T)
for ax, (label, (vx, _)) in zip(axes.flat, snapshots.items()):
    ax.hist(vx, bins=45, range=(-3.5, 3.5), density=True, alpha=0.7)
    ax.plot(v, gauss, "k--", lw=1.2, label="Gaussian, measured T")
    ax.set_title(label)
    ax.legend(fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("$v_x$")
for ax in axes[:, 0]:
    ax.set_ylabel("probability density")
fig.suptitle("A Gaussian velocity distribution assembles itself")
fig.tight_layout()
plt.show()

## Speeds: the 2D Maxwell–Boltzmann distribution

Same data, but now the speed $|v|$. It starts as a delta function at
$v_0$ and relaxes to the 2D Maxwell–Boltzmann form
$f(v) = (v/T)\,e^{-v^2/2T}$:

In [ ]:
v = np.linspace(0, 4, 300)
mb2d = (v / T) * np.exp(-v**2 / (2 * T))

plt.figure(figsize=(6, 3.6))
plt.hist(snapshots["t = 0"][1], bins=45, range=(0, 4), density=True,
         alpha=0.55, label="t = 0 (all speeds = 1.5)")
plt.hist(snapshots["after 4000 steps"][1], bins=45, range=(0, 4), density=True,
         alpha=0.55, label="after 4000 steps")
plt.plot(v, mb2d, "k--", lw=1.2, label="2D Maxwell–Boltzmann")
plt.xlabel("speed $|v|$"); plt.ylabel("probability density")
plt.legend(); plt.tight_layout(); plt.show()

lmp.close()

Every collision conserved energy and momentum exactly, yet the initial
condition was forgotten completely — only its total energy survives, as the
temperature setting the width of the Gaussian.

Next: [02 — Condensation](02-condensation.ipynb), where the same gas is
quenched below its condensation point and droplets rain out.